# SQL Analysis for ChromaDB

> **`Chromadb sqlite3` 기반 스키마 확인 및 SQL 질의 분석**


`rag_sql_analysis.ipynb`를 쓰면 좋은 경우:
- Chroma에 어떤 `collection`이 실제로 들어갔는지 확인할 때
- `source`, `name`, `section_title` 같은 metadata 분포를 점검할 때
- retriever filter가 먹을 수 있는 필드가 실제로 있는지 검증할 때
- RAG 결과가 이상할 때, 문제 원인이 `문서 적재`인지 `검색 로직`인지 분리해서 볼 때
- 운영 중 DB 상태를 샘플링해서 점검할 때

굳이 안 써도 되는 경우:
- 최종 사용자 질문에 답변하는 RAG 실행
- 프롬프트 품질 비교
- retrieval 파라미터 튜닝 실험
- 실제 데모/서비스 흐름 검증

### 환경 구축

In [1]:
# API Key 가져오기 (서비스 개시 전, Github secrets key로 전환)

import os
import pandas as pd
import sqlite3

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain.agents import create_agent
from pathlib import Path
from pydantic import BaseModel, Field



load_dotenv('.env')     # 같은 폴더의 .env
load_dotenv('../.env')  # 상위 폴더의 .env

if not os.getenv('OPENAI_API_KEY') :
    raise RuntimeError(
        'OPENAI_API_KEY를 찾지 못했습니다.\n'
        '다시 확인해 주세요.'
    )
else :
    model = ChatOpenAI(model='gpt-4o-mini', temperature=0, timeout=60)  # 서비스 개시 전, temperature 변경
    print("OpenAI API Key 확인 하고 'MODEL' 생성\n", model.get_graph)

d:\encore\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI API Key 확인 하고 'MODEL' 생성
 <bound method Runnable.get_graph of ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.3'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000026789BB0590>, async_client=<openai.resources.chat.completions.completions.AsyncCompleti

### ChromaDB CONNECTION

In [2]:
DB_PATH = Path('../chroma_db') / 'chroma.sqlite3'
DB_PATH.parent.mkdir(exist_ok=True)

_conn = sqlite3.connect(DB_PATH, isolation_level=None)

_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, isolation_level=None, check_same_thread=False)

print('데이터베이스 연결 완료', DB_PATH)

데이터베이스 연결 완료 ..\chroma_db\chroma.sqlite3


### QUERY 결과 → DataFrame 변환

In [3]:
def run_query(sql) :
    """SELECT 결과를 DataFrame 으로 변환"""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


### QUERY TEST

In [4]:
display(run_query('select * from embeddings'))

,id,segment_id,embedding_id,seq_id,created_at
0,1,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_0,1,2026-08-23 04:41:47
1,2,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_1,2,2026-08-23 04:41:47
2,3,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_2,3,2026-08-23 04:41:47
3,4,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3,4,2026-08-23 04:41:47
4,5,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_4,5,2026-08-23 04:41:47
...,...,...,...,...,...
3689,3690,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3689,3690,2026-08-23 04:41:50
3690,3691,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3690,3691,2026-08-23 04:41:50
3691,3692,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3691,3692,2026-08-23 04:41:50
3692,3693,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3692,3693,2026-08-23 04:41:50


### Query System prompt

In [5]:
schema_prompt = """You are an assistant that understands the schema of a ChromaDB SQLite database.

The database schema is based on an actual Chroma SQLite file and should be interpreted as follows:

Core hierarchy:
- tenants: top-level tenant records
- databases: belongs to a tenant
- collections: belongs to a database, represents a vector collection
- segments: belongs to a collection, represents internal collection segments
- embeddings: belongs logically to a segment, represents individual embedding records

Tables and meanings:

1) tenants
- id TEXT PRIMARY KEY
- Tenant identifier

2) databases
- id TEXT PRIMARY KEY
- name TEXT NOT NULL
- tenant_id TEXT NOT NULL REFERENCES tenants(id) ON DELETE CASCADE
- UNIQUE (tenant_id, name)
- A tenant can own multiple databases

3) collections
- id TEXT PRIMARY KEY
- name TEXT NOT NULL
- dimension INTEGER
- database_id TEXT NOT NULL REFERENCES databases(id) ON DELETE CASCADE
- config_json_str TEXT
- schema_str TEXT
- UNIQUE (name, database_id)
- Represents a vector collection

4) collection_metadata
- collection_id TEXT REFERENCES collections(id) ON DELETE CASCADE
- key TEXT NOT NULL
- str_value TEXT
- int_value INTEGER
- float_value REAL
- bool_value INTEGER
- PRIMARY KEY (collection_id, key)
- Key-value metadata for collections

5) segments
- id TEXT PRIMARY KEY
- type TEXT NOT NULL
- scope TEXT NOT NULL
- collection TEXT NOT NULL
- The DDL literally says: collection TEXT REFERENCES collection(id) NOT NULL
- Semantically, this should be treated as the collection reference for the segment
- Note that the literal FK target in the DDL appears inconsistent with the actual collections table name

6) segment_metadata
- segment_id TEXT REFERENCES segments(id) ON DELETE CASCADE
- key TEXT NOT NULL
- str_value TEXT
- int_value INTEGER
- float_value REAL
- bool_value INTEGER
- PRIMARY KEY (segment_id, key)
- Key-value metadata for segments

7) embeddings
- id INTEGER PRIMARY KEY
- segment_id TEXT NOT NULL
- embedding_id TEXT NOT NULL
- seq_id BLOB NOT NULL
- created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
- UNIQUE (segment_id, embedding_id)
- Main record for an embedding entry

8) embedding_metadata
- id INTEGER REFERENCES embeddings(id)
- key TEXT NOT NULL
- string_value TEXT
- int_value INTEGER
- float_value REAL
- bool_value INTEGER
- PRIMARY KEY (id, key)
- Single-valued metadata attached to an embedding

9) embedding_metadata_array
- id INTEGER NOT NULL REFERENCES embeddings(id)
- key TEXT NOT NULL
- string_value TEXT
- int_value INTEGER
- float_value REAL
- bool_value INTEGER
- Multi-valued or array-style metadata attached to an embedding

10) embeddings_queue
- seq_id INTEGER PRIMARY KEY
- created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
- operation INTEGER NOT NULL
- topic TEXT NOT NULL
- id TEXT NOT NULL
- vector BLOB
- encoding TEXT
- metadata TEXT
- Queue/event table for embedding operations

11) embeddings_queue_config
- id INTEGER PRIMARY KEY
- config_json_str TEXT

12) max_seq_id
- segment_id TEXT PRIMARY KEY
- seq_id INTEGER
- Tracks max sequence per segment

13) acquire_write
- id INTEGER PRIMARY KEY
- lock_status INTEGER NOT NULL
- Write-lock coordination table

14) maintenance_log
- id INT PRIMARY KEY
- timestamp INT NOT NULL
- operation TEXT NOT NULL

15) migrations
- dir TEXT NOT NULL
- version INTEGER NOT NULL
- filename TEXT NOT NULL
- sql TEXT NOT NULL
- hash TEXT NOT NULL
- PRIMARY KEY (dir, version)

Full-text search:
- embedding_fulltext_search is an FTS5 virtual table on string_value using trigram tokenization
- Related internal FTS tables also exist:
  embedding_fulltext_search_config
  embedding_fulltext_search_content
  embedding_fulltext_search_data
  embedding_fulltext_search_docsize
  embedding_fulltext_search_idx

Important interpretation rules:
- Treat collections, segments, embeddings, and metadata tables as the main functional schema.
- Treat FTS helper tables as implementation details unless the user specifically asks about full-text indexing internals.
- When explaining relationships, use the semantic relationship even if the literal DDL has minor inconsistencies.
- If generating SQL or analysis, be explicit when a relationship is inferred semantically rather than strictly enforced by the visible DDL.
- bool_value fields are stored as INTEGER.
- Metadata values are polymorphic and split across string/int/float/bool columns.

When answering:
- Creates only a single SELECT statement.
- Organizes the results into human-readable Korean sentences and provides the answer.
- Prefer clear schema-aware explanations.
- If asked to write SQL, base it on this schema.
- If there is ambiguity, mention whether you are following literal DDL or semantic intent.
"""


### Query 검사

In [6]:
@tool
def run_select(sql : str) -> str :
    """SELECT 문을 실행하고 결과를 문자열로 반환한다."""
    stmt = sql.strip().rstrip(';')

    # Query 문에 'select' 이외 다른 실행문은 없는지 검사해서 있으면 차단!
    if not stmt.lower().startswith('select') or ';' in stmt :
        return '중단 : SELECT 문 이외 실행은 불가능 합니다.'
    try : 
        return str(_ro_conn.execute(stmt).fetchall())
    except Exception as e :
        return f'에러 : {e}'


### Query Agent 생성

In [7]:
sql_agent = create_agent(model, [run_select], system_prompt=schema_prompt)

### 구조화된 출력

In [8]:
class SqlForm(BaseModel) :
    """자연어 질문에 대한 결과를 구조화된 형식으로 받는다."""
    sql : str = Field(description = '실행할 SELECT문 표시')
    reason : str = Field(description = 'Query 실행 결과에 대한 이유를 한 문장으로 표시')
    tables : list[str] = Field(description = 'Query가 사용하는 표 이름 목록 표시')
    question : str = Field(decription = '질문 내용')
    answer : str = Field(description = '모델의 최종 답변')

print('스키마 필드 : ', list(SqlForm.model_fields))

스키마 필드 :  ['sql', 'reason', 'tables', 'question', 'answer']


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24968\724729884.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  question : str = Field(decription = '질문 내용')


### OpenAI API + BaseModel

In [9]:
question = input('질문을 입력해 주세요. : ')

query_result = model.with_structured_output(SqlForm).invoke(schema_prompt + question)

In [10]:
# 질문에 대한 답변을 구조화된 출력으로 확인해 본다.
print('결과의 종류 : ', type(query_result).__name__)
print('쿼리 문 : ', query_result.sql)
print('이유 : ', query_result.reason)
print('테이블 : ', query_result.tables)
print('질문 : ', query_result.question)
print('답변 : ', query_result.answer)

결과의 종류 :  SqlForm
쿼리 문 :  SELECT name FROM collections WHERE dimension > 0;
이유 :  이 쿼리는 차원이 0보다 큰 모든 컬렉션의 이름을 가져옵니다. 이는 유효한 벡터 컬렉션을 찾기 위한 것입니다.
테이블 :  ['collections']
질문 :  차원이 0보다 큰 모든 컬렉션의 이름을 알고 싶어요.
답변 :  차원이 0보다 큰 모든 컬렉션의 이름을 가져오는 쿼리입니다.
